In [1]:
import pandas as pd
import numpy as np

In [2]:
df_train = pd.read_csv("../../data/processed/train_engineered.csv")
df_val   = pd.read_csv("../../data/processed/val_engineered.csv")
df_test  = pd.read_csv("../../data/processed/test_engineered.csv")

In [3]:
FEATURE_COLS = [
    # Time
    'hour',
    'day_of_week',
    'is_night',

    # Amount
    'log_amount',
    'amount_to_card_mean',

    # Email
    'email_domain_mismatch',
    'purchaser_email_risk',
    'p_email_freq',

    # Device
    'is_mobile',

    # Card
    'card4_freq',
    'card6_freq'
] + [c for c in df_train.columns if c.startswith('prod_')]

In [4]:
X_train = df_train[FEATURE_COLS].fillna(0)
y_train = df_train['isFraud']

X_val = df_val[FEATURE_COLS].fillna(0)
y_val = df_val['isFraud']

X_test = df_test[FEATURE_COLS].fillna(0)
y_test = df_test['isFraud']

In [5]:
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("http://127.0.0.1:5000")
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import(roc_auc_score,average_precision_score,precision_recall_curve,classification_report)
import joblib


In [6]:
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train)
X_val_sc=scaler.transform(X_val)
X_test_sc=scaler.transform(X_test)

mlflow.set_experiment("fraud-detection-ieee-cis")
with mlflow.start_run(run_name="random_forest_classifier"):
    mlflow.log_param("model","RandomForest")
    mlflow.log_param("n_estimators",200)
    mlflow.log_param("class_weight","balanced")
    mlflow.log_param("feature_count",len(FEATURE_COLS))

    rf=RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=35,n_jobs=-1)
    rf.fit(X_train_sc,y_train)
    y_prob_val_rf=rf.predict_proba(X_val_sc)[:,1]
    pr_auc_rf=average_precision_score(y_val,y_prob_val_rf)
    roc_auc_rf=roc_auc_score(y_val,y_prob_val_rf)
    mlflow.log_metric("val_pr_auc",pr_auc_rf)
    mlflow.log_metric("val_roc_auc",roc_auc_rf)
    mlflow.sklearn.log_model(rf,'random_forest')
    print(f"RF Baseline — PR-AUC: {pr_auc_rf:.4f} | ROC-AUC: {roc_auc_rf:.4f}")



2026/06/02 16:07:53 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-ieee-cis' does not exist. Creating a new experiment.
2026/06/02 16:08:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 16:08:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF Baseline — PR-AUC: 0.1194 | ROC-AUC: 0.7306
🏃 View run random_forest_classifier at: http://127.0.0.1:5000/#/experiments/1/runs/80592d64f45a4b3587dc7dd6c5a7a85b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
